<a href="https://colab.research.google.com/github/pallavmarch/Psychologytoday-therapist-analysis/blob/main/Xlookup_new_links.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This Python script is used to identify mismatches or missing therapist profiles in a data collection process. Specifically, it finds profile URLs listed in a link file that do not have corresponding names in the main data file. This is useful for cleaning, validating, or troubleshooting web-scraped datasets

In [5]:
import pandas as pd
from google.colab import files

##### texas | california | florida | new-york
selected_state = "new-york"


state_files = {
    selected_state: {
        "main_file": f"/content/therapist_data_{selected_state}.csv",
        "link_file": f"/content/therapist_link_{selected_state}.csv",
        "output_file": f"missing_names_{selected_state}.csv",
    }
}

main_file_path = state_files[selected_state]["main_file"]
link_file_path = state_files[selected_state]["link_file"]
output_file_path = state_files[selected_state]["output_file"]

df_main = pd.read_csv(main_file_path)
df_link = pd.read_csv(link_file_path)

df_main=df_main[['Profile URL','Name']]
df_link = df_link.merge(df_main, on='Profile URL', how='left')

print(f"Total rows in {link_file_path}: {df_link.shape[0]}")
print(f"Total null values in 'Name' column: {df_link['Name'].isnull().sum()}")
df_link = df_link[df_link['Name'].isnull()]

df_link.to_csv(output_file_path, index=False)
files.download(output_file_path)

Total rows in /content/therapist_link_new-york.csv: 487
Total null values in 'Name' column: 291


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
file = "/content/missing_names_texas.csv"

import pandas as pd
from google.colab import files
from bs4 import BeautifulSoup
import requests
import time
import random
from tqdm import tqdm


df_link=pd.read_csv(file)
current_profiles=df_link['Profile URL'].tolist()


HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}


all_therapists_data = []



def extract_specialties_expertise(soup):

    data = {key: "Not Found" for key in ["Top Specialties", "Expertise"]}
    specialty_section = soup.find("div", id="specialty-attributes-section")

    if specialty_section:
        for group in specialty_section.find_all("div", class_="attributes-group"):
            heading = group.find("h3")

            if heading:
                key = "Top Specialties" if "Top Specialties" in heading.get_text(strip=True) else "Expertise"

                data[key] = " | ".join(span.get_text(strip=True).lower() for span in group.find_all("span", class_="attribute_base"))

    return data


for idx, url in enumerate(tqdm(current_profiles, desc="Scraping Progress", colour="red", ncols=100,unit="profile")):

    try:
        response = requests.get(url, headers=HEADERS)
        time.sleep(random.uniform(1, 2))

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")


            therapist_data = {
                **extract_specialties_expertise(soup)
            }

            all_therapists_data.append(therapist_data)

    except Exception as e:
        print(f"⚠️ Error scraping {url}: {e}")


df = pd.DataFrame(all_therapists_data)
#df.to_csv(file, index=False)
#files.download(file)


Scraping Progress:  39%|██████████████▋                       | 72/186 [02:23<03:47,  2.00s/profile]


KeyboardInterrupt: 